In [1]:
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.transforms as T
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import negative_sampling, to_undirected
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device", device)

d:\WIA\CLASS\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\WIA\CLASS\.env\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Device cpu


In [2]:
# ---------------------------------------------------------------
# 1. CSV -> graph (nodes = IPs, edges = "src->dst" rows)
# ---------------------------------------------------------------
df = pd.read_csv("data(2).csv")   # <-- change path if needed

feat_cols = [c for c in df.columns if c not in ("node_id", "attack")]
for c in feat_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")      # '#', '*', 'a' -> NaN
df[feat_cols] = df[feat_cols].fillna(df[feat_cols].median())

sd = df["node_id"].str.split("->", expand=True)
df["src"], df["dst"] = sd[0], sd[1]
ips = sorted(set(df["src"]) | set(df["dst"]))
ip2idx = {ip: i for i, ip in enumerate(ips)}
src = torch.tensor(df["src"].map(ip2idx).values, dtype=torch.long)
dst = torch.tensor(df["dst"].map(ip2idx).values, dtype=torch.long)
N = len(ips)
edge_index = to_undirected(torch.stack([src, dst]), num_nodes=N)

In [3]:
# Node features: mean of flow features going out of / into each IP + degrees
E = df[feat_cols].values.astype(float)
E = np.sign(E) * np.log1p(np.abs(E))
E = torch.tensor(E, dtype=torch.float)

def mean_by_node(index):
    total = torch.zeros(N, E.size(1)).index_add_(0, index, E)
    count = torch.zeros(N).index_add_(0, index, torch.ones(len(index)))
    return total / count.clamp(min=1).unsqueeze(1), count

out_feat, out_cnt = mean_by_node(src)
in_feat, in_cnt = mean_by_node(dst)
X = torch.cat([out_feat, in_feat,
               out_cnt.log1p().unsqueeze(1), in_cnt.log1p().unsqueeze(1)], dim=1)
X = torch.tensor(StandardScaler().fit_transform(X.numpy()), dtype=torch.float)

data = Data(x=X, edge_index=edge_index)
print(data)
print("Nodes:", data.num_nodes)
print("Edges:", data.edge_index.shape[1])
print("Features:", data.x.shape)


Data(x=[93, 24], edge_index=[2, 262])
Nodes: 93
Edges: 262
Features: torch.Size([93, 24])


In [4]:
# 2. Link split
# ---------------------------------------------------------------
torch.manual_seed(0)
split_transform = T.RandomLinkSplit(num_val=0.1, num_test=0.2,
                                    is_undirected=True,
                                    add_negative_train_samples=False)
train_data, val_data, test_data = split_transform(data)
print(train_data)
print(val_data)
print(test_data)

Data(x=[93, 24], edge_index=[2, 184], edge_label=[92], edge_label_index=[2, 92])
Data(x=[93, 24], edge_index=[2, 184], edge_label=[26], edge_label_index=[2, 26])
Data(x=[93, 24], edge_index=[2, 210], edge_label=[52], edge_label_index=[2, 52])


In [5]:
# ---------------------------------------------------------------
# 3. Model
# ---------------------------------------------------------------
class GCNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(data.num_features, 64)
        self.conv2 = GCNConv(64, 32)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        z = self.conv2(x, edge_index)
        return z

    def decode(self, z, edge_label_index):
        source = edge_label_index[0]
        destination = edge_label_index[1]
        scores = (z[source] * z[destination]).sum(dim=-1)
        return scores

train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)
all_original_edges = data.edge_index.to(device)

model = GCNeuralNetwork().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()
print(model)

GCNeuralNetwork(
  (conv1): GCNConv(24, 64)
  (conv2): GCNConv(64, 32)
)


In [6]:

# ---------------------------------------------------------------
# 4. Train / evaluate functions
# ---------------------------------------------------------------
def train():
    model.train()
    optimizer.zero_grad()
    z = model.encode(train_data.x, train_data.edge_index)
    pos_edge_index = train_data.edge_label_index
    pos_scores = model.decode(z, pos_edge_index)
    neg_edge_index = negative_sampling(
        edge_index=all_original_edges,
        num_nodes=data.num_nodes,
        num_neg_samples=pos_edge_index.size(1),
        method="sparse"
    )
    neg_scores = model.decode(z, neg_edge_index)
    scores = torch.cat([pos_scores, neg_scores])
    labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
    loss = criterion(scores, labels)
    loss.backward()
    optimizer.step()
    return loss.item()




In [7]:
@torch.no_grad()
def evaluate(split_data):
    model.eval()
    z = model.encode(split_data.x, split_data.edge_index)
    scores = model.decode(z, split_data.edge_label_index)
    y_score = torch.sigmoid(scores).cpu().numpy()
    y_true = split_data.edge_label.cpu().numpy()
    auc = roc_auc_score(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    return auc, ap

In [8]:

# ---------------------------------------------------------------
# 5. Training loop
# ---------------------------------------------------------------
epochs = 100
for epoch in range(1, epochs + 1):
    loss = train()
    val_auc, val_ap = evaluate(val_data)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} Loss:{loss:.4f} Val AUC:{val_auc:.4f} Val AP:{val_ap:.4f}")
test_auc, test_ap = evaluate(test_data)
print("\nFinal Test Results")
print(f"Test ROC-AUC:{test_auc:.4f}")
print(f"Test Average Precision: {test_ap:.4f}")

Epoch 010 Loss:0.4765 Val AUC:0.8639 Val AP:0.9215
Epoch 020 Loss:0.4145 Val AUC:0.8402 Val AP:0.9151
Epoch 030 Loss:0.3812 Val AUC:0.7219 Val AP:0.8383
Epoch 040 Loss:0.4842 Val AUC:0.5503 Val AP:0.7463
Epoch 050 Loss:0.3731 Val AUC:0.5385 Val AP:0.7433
Epoch 060 Loss:0.4283 Val AUC:0.5562 Val AP:0.7478
Epoch 070 Loss:0.3827 Val AUC:0.6391 Val AP:0.7836
Epoch 080 Loss:0.4819 Val AUC:0.5385 Val AP:0.7433
Epoch 090 Loss:0.4227 Val AUC:0.5621 Val AP:0.7496
Epoch 100 Loss:0.4139 Val AUC:0.7633 Val AP:0.8590

Final Test Results
Test ROC-AUC:0.5962
Test Average Precision: 0.7592
